In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Audio, display
from pynwb import NWBHDF5IO
from collections import Counter
import glob
from datetime import datetime

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, IntSlider, FloatSlider, Dropdown, Checkbox

from brain_audio_decoder import BrainAudioDecoder
from custom_decoder import CustomBrainAudioDecoder
from brain_audio_decoder_viz import BrainAudioDecoderViz
from acoustic_change_detector import AcousticChangeDetector
from phoneme_validator import PhonemeValidator
from phonetic_dictionary import PhoneticDictionary
from hybrid_phoneme_models import HybridPhonemeModels
from pipeline import UnifiedPhonemePipeline
from markov_phoneme_model import MarkovPhonemeModel
from markov_children import  GMMHMMModel, HSMMModel
from diverse_models import SimplePhonemeModels

In [2]:
# Define paths
path_bids = './SingleWordProductionDutch-iBIDS'
path_output = './features'
path_results = './results'

# Create directories if they don't exist
os.makedirs(path_output, exist_ok=True)
os.makedirs(path_results, exist_ok=True)

In [3]:
# Pipeline creation with PCA components management
use_augmentation = True
feature_extraction_method = 'high_gamma'  # 'multi_band'
optimal_pca_components = 50  # Use your optimal value determined earlier
use_phoneme_groups = True 

# Try to load existing pipeline, otherwise create new one
try:
    # Try loading existing pipeline
    pipeline = UnifiedPhonemePipeline.load_saved(path_results, method=feature_extraction_method)
    print(f"Loaded existing {feature_extraction_method} pipeline")
    
    # Check and update PCA components if needed
    current_pca = getattr(pipeline, 'pca_components', None)
    if current_pca != optimal_pca_components:
        print(f"Updating PCA components from {current_pca} to {optimal_pca_components}")
        pipeline.set_pca_components(optimal_pca_components)
        
        # Re-run data steps with new PCA components
        print("Re-processing data with updated PCA components...")
        pipeline.step4_initialize_detector()    
        pipeline.step5_accumulate_data()
        pipeline.step6_resolve_unknowns()
        pipeline.step7_filter_unknowns()
        
        # Save the updated pipeline
        pipeline.save()
        print(f"Updated and saved {feature_extraction_method} pipeline with {optimal_pca_components} PCA components")
    
except (FileNotFoundError, AttributeError, TypeError) as e:
    # No existing pipeline found, create new one
    print(f"No existing {feature_extraction_method} pipeline found. Creating new one...")
    
    pipeline = UnifiedPhonemePipeline(
        path_bids=path_bids,
        path_output=path_output,
        path_results=path_results,
        feature_extraction_method=feature_extraction_method,
        unknown_keep_ratio=0.1,
        channel_correlation_threshold=0.3,  # ADD THIS
        prioritize_regions=True,  # ADD THIS
        channel_selection='best_correlation',
        pca_components=optimal_pca_components,  # Set optimal PCA components
        debug_mode=False
    )
    
    # Run all steps
    print("Running pipeline steps...")
    pipeline.step1_initialize_decoder()
    pipeline.step2_stratify_participants(
                            channel_correlation_threshold=0.3,
                            prioritize_regions=True)    
    pipeline.step3_create_split()
    pipeline.step4_initialize_detector()    
    pipeline.step5_accumulate_data()
    pipeline.step6_resolve_unknowns()
    pipeline.step7_filter_unknowns()
    
    # Save the pipeline
    pipeline.save()
    print(f"Created and saved new {feature_extraction_method} pipeline with {optimal_pca_components} PCA components")

# The pipeline is now ready to use with optimal PCA components
print(f"Pipeline ready with {feature_extraction_method} features and {optimal_pca_components} PCA components")

Loading newest matching pipeline: pipeline_high_gamma_pca50_20250821_201128.pkl
PhoneticDictionary: Initialized with DEBUG_MODE=False
UnifiedPhonemePipeline: Pipeline initialized: high_gamma, PCA=50, groups=True
UnifiedPhonemePipeline: Loaded high_gamma pipeline with PCA components=50
UnifiedPhonemePipeline: Train: 160 samples, Test: 96 samples
Loaded existing high_gamma pipeline
Pipeline ready with high_gamma features and 50 PCA components


In [4]:
# # Get training and test data
# train_data = pipeline.get_training_data(filtered=pipeline.get_training_data(filtered=True))
# test_data = pipeline.get_test_data()
    
# # Check if we have data
# if not train_data or 'features' not in train_data or not train_data['features']:
#     print("Error: No training data available")
    
# # Initialize the models
# simple_models = SimplePhonemeModels(debug_mode=True)
    
# # Train on individual phonemes (no need to map to groups)
# training_results = simple_models.train_all_classical(
#         train_data['features'], 
#         train_data['phoneme_labels']
#     )
    
# # Evaluate on test data
# evaluation_results = {}
# for model_name in simple_models.models:
#     try:
#             # Make predictions
#         predictions, _ = simple_models.predict(
#                 test_data['features'], 
#                 model_name
#             )
            
#             # Calculate accuracy
#         accuracy = np.mean(predictions == test_data['phoneme_labels'])
#         evaluation_results[model_name] = accuracy
            
#         print(f"{model_name} test accuracy: {accuracy:.4f}")
#     except Exception as e:
#         print(f"Error evaluating {model_name}: {e}")

In [5]:
import gc
import traceback

def compare_feature_extraction_methods(pipeline_class, path_bids, path_output, path_results,
                                     methods=None, pca_components=30, 
                                     use_phoneme_groups=True,
                                     debug_mode=True):
    """Compare different feature extraction methods for phoneme classification."""
    
    if methods is None:
        methods = ['high_gamma', 'multi_band', 'spectral', 'wavelet', 'mfcc']
    
    results = {}
    
    # Create a timestamped file for results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_file = os.path.join(path_results, f"feature_extraction_comparison_{timestamp}.txt")
    
    # Write header
    with open(results_file, 'w') as f:
        f.write(f"Feature Extraction Method Comparison - {timestamp}\n")
        f.write(f"PCA Components: {pca_components}\n")
        f.write(f"Use Phoneme Groups: {use_phoneme_groups}\n")
        f.write("="*60 + "\n\n")
    
    # Create base pipeline for common components
    base_pipeline = None
    
    for method in methods:
        print(f"\n{'='*50}")
        print(f"Testing feature extraction method: {method}")
        print(f"{'='*50}\n")
        
        # Log to file
        with open(results_file, 'a') as f:
            f.write(f"\n{'='*50}\n")
            f.write(f"Testing feature extraction method: {method}\n")
            f.write(f"{'='*50}\n\n")
        
        # INITIALIZE VARIABLES BEFORE TRY BLOCK
        train_data = None
        test_data = None
        balanced_train = None
        
        try:
            # 1. CREATE OR LOAD PIPELINE
            pipeline = pipeline_class(
                path_bids=path_bids,
                path_output=path_output,
                path_results=path_results,
                feature_extraction_method=method,
                pca_components=pca_components,
                use_phoneme_groups=use_phoneme_groups,
                channel_correlation_threshold=0.3,  # ADD THIS
                prioritize_regions=True,  # ADD THIS
                channel_selection='best_correlation', 
                debug_mode=debug_mode
            )
            
            # Try to load checkpoint
            checkpoint_loaded = pipeline.try_load_checkpoint()
            
            if checkpoint_loaded:
                print(f"Loaded checkpoint for {method}")
                # Ensure phonetic dictionary exists
                if not hasattr(pipeline, 'phonetic_dict'):
                    from phonetic_dictionary import PhoneticDictionary
                    pipeline.phonetic_dict = PhoneticDictionary()
                    pipeline.phonetic_dict.add_phoneme_groups()
                
                # Ensure filtered data exists
                if not hasattr(pipeline, 'train_filtered'):
                    pipeline.step7_filter_unknowns()
                    
                # IMPORTANT: Run step 8 if using groups and checkpoint was loaded
                if use_phoneme_groups:
                    pipeline.step8_convert_to_groups()
            else:
                # No checkpoint - run pipeline
                if base_pipeline is None:
                    # First method - run full pipeline (includes step 8)
                    print(f"Creating new pipeline for {method}")
                    pipeline.run_all_steps()
                    base_pipeline = pipeline
                else:
                    # Reuse base components for efficiency
                    print(f"Reusing base components for {method}")
                    pipeline.participant_strata = base_pipeline.participant_strata
                    pipeline.split_result = base_pipeline.split_result
                    pipeline.phonetic_dict = base_pipeline.phonetic_dict
                    
                    # Run method-specific steps
                    pipeline.step1_initialize_decoder()
                    if not pipeline.run_steps_4_to_7():
                        raise ValueError(f"Failed to run steps 4-7 for {method}")
                    
                    # Don't forget step 8!
                    if use_phoneme_groups:
                        pipeline.step8_convert_to_groups()
            
            # 2. GET AND VALIDATE DATA
            train_data = pipeline.get_training_data(filtered=True)
            test_data = pipeline.get_test_data()
            
            if not train_data or not train_data.get('features'):
                raise ValueError(f"No training data for {method}")
            if not test_data or not test_data.get('features'):
                raise ValueError(f"No test data for {method}")
            
            # DIAGNOSTIC: Check label distributions
            from collections import Counter
            train_labels_dist = Counter(train_data['phoneme_labels'])
            test_labels_dist = Counter(test_data['phoneme_labels'])
            
            print(f"Train unique labels: {len(train_labels_dist)}")
            print(f"Test unique labels: {len(test_labels_dist)}")
            print(f"Train label sample: {list(train_labels_dist.keys())[:5]}")
            print(f"Test label sample: {list(test_labels_dist.keys())[:5]}")
            
            # Log data info
            print(f"Train: {len(train_data['features'])} samples, "
                  f"Test: {len(test_data['features'])} samples")
            
            with open(results_file, 'a') as f:
                f.write(f"Train data: {len(train_data['features'])} samples\n")
                f.write(f"Test data: {len(test_data['features'])} samples\n")
                f.write(f"Train unique labels: {len(train_labels_dist)}\n")
                f.write(f"Test unique labels: {len(test_labels_dist)}\n\n")
            
            # 3. BALANCE TRAINING DATA
            balanced_train = pipeline.balance_training_data(
                train_data, 
                balance_strategy='oversample'
            )
            print(f"Balanced training data: {len(balanced_train['features'])} samples")
            
            # VERIFY BALANCING
            balanced_dist = Counter(balanced_train['phoneme_labels'])
            print(f"Balanced distribution: {dict(balanced_dist)}")
            
            # Only proceed if balancing succeeded
            if not balanced_train or not balanced_train.get('features'):
                raise ValueError(f"Balancing failed for {method}")
            
            # 4. TRAIN AND EVALUATE MODELS
            # Create fresh model instance to avoid shape conflicts
            simple_models = SimplePhonemeModels(debug_mode=debug_mode)
            # Clear any cached models/scalers/encoders
            simple_models.models = {}
            simple_models.scalers = {}
            simple_models.label_encoders = {}
            
            # Train all models
            print("Training models")
            training_results = simple_models.train_all_models(
                balanced_train['features'], 
                balanced_train['phoneme_labels'], 
                include_neural=True
            )
            
            # Evaluate all models
            print("Evaluating models")
            eval_results = simple_models.evaluate_all_models(
                test_data['features'], 
                test_data['phoneme_labels']
            )
            
            # Get unique labels from the data
            unique_labels = sorted(set(test_data['phoneme_labels']))

            # Create confusion matrices for top models
            for model_name in eval_results.keys():
                if model_name in simple_models.models and eval_results.get(model_name):
                    try:
                        predictions, _ = simple_models.predict(test_data['features'], model_name)
                        cm = confusion_matrix(test_data['phoneme_labels'], predictions, labels=unique_labels)

                        # Plot with labels and numbers
                        plt.figure(figsize=(12, 10))
                        plt.imshow(cm, interpolation='nearest', cmap='Blues')
                        plt.title(f'{method} - {model_name} (Acc: {eval_results[model_name]:.3f})')
                        plt.colorbar()

                        # Add labels
                        tick_marks = np.arange(len(unique_labels))
                        plt.xticks(tick_marks, unique_labels, rotation=45, ha='right')
                        plt.yticks(tick_marks, unique_labels)

                        # Add numbers in cells
                        thresh = cm.max() / 2.
                        for i in range(cm.shape[0]):
                            for j in range(cm.shape[1]):
                                plt.text(j, i, format(cm[i, j], 'd'),
                                        ha="center", va="center",
                                        color="white" if cm[i, j] > thresh else "black")

                        plt.ylabel('True Label')
                        plt.xlabel('Predicted Label')
                        plt.tight_layout()
                        plt.savefig(os.path.join(path_results, f'cm_{method}_{model_name}_{timestamp}.png'))
                        plt.close()  # Close to avoid memory issues

                    except Exception as e:
                        print(f"Could not create confusion matrix for {model_name}: {e}")
            
            
            # DIAGNOSTIC: Check one model's predictions
            if 'naive_bayes' in simple_models.models:
                test_predictions, _ = simple_models.predict(test_data['features'][:10], 'naive_bayes')
                print(f"Sample predictions: {test_predictions}")
                print(f"Sample true labels: {test_data['phoneme_labels'][:10]}")
            
            # 5. RECORD RESULTS
            method_results = {}
            with open(results_file, 'a') as f:
                f.write("Model Evaluation Results:\n")
                f.write("-"*30 + "\n")
                
                for model_name, accuracy in eval_results.items():
                    if accuracy is not None:
                        method_results[model_name] = accuracy
                        print(f"  {model_name}: {accuracy:.4f}")
                        f.write(f"{model_name}: {accuracy:.4f}\n")
                
                # Best model for this method
                if method_results:
                    best_model = max(method_results, key=method_results.get)
                    best_acc = method_results[best_model]
                    f.write(f"\nBest: {best_model} ({best_acc:.4f})\n\n")
                    print(f"Best for {method}: {best_model} ({best_acc:.4f})")
            
            results[method] = method_results
            
            del pipeline
            gc.collect()
            
        except Exception as e:
            print(f"Error processing {method}: {e}")
            
            # Debug info - now these variables exist
            if train_data:
                print(f"  Had training data: {len(train_data.get('features', []))} samples")
            if test_data:
                print(f"  Had test data: {len(test_data.get('features', []))} samples")
            if balanced_train:
                print(f"  Had balanced data: {len(balanced_train.get('features', []))} samples")            
            with open(results_file, 'a') as f:
                f.write(f"Error: {e}\n\n")

            traceback.print_exc()
    
    # 6. FINAL SUMMARY
    print("\n" + "="*60)
    print("FINAL COMPARISON SUMMARY")
    print("="*60)
    
    with open(results_file, 'a') as f:
        f.write("\n" + "="*60 + "\n")
        f.write("FINAL COMPARISON SUMMARY\n")
        f.write("="*60 + "\n")
        
        if results:
            # Find best overall
            best_method = None
            best_model = None
            best_acc = 0
            
            for method, method_results in results.items():
                if method_results:
                    method_best = max(method_results, key=method_results.get)
                    method_acc = method_results[method_best]
                    
                    print(f"{method:10s}: {method_acc:.4f} ({method_best})")
                    f.write(f"{method:10s}: {method_acc:.4f} ({method_best})\n")
                    
                    if method_acc > best_acc:
                        best_acc = method_acc
                        best_method = method
                        best_model = method_best
            
            if best_method:
                print(f"\nOVERALL BEST: {best_method} with {best_model} ({best_acc:.4f})")
                f.write(f"\nOVERALL BEST: {best_method} with {best_model} ({best_acc:.4f})\n")
        else:
            print("No valid results")
            f.write("No valid results\n")
    
    print(f"\nResults saved to: {results_file}")
    return results, results_file

In [6]:
# Run the comparison
extraction_results = compare_feature_extraction_methods(
    UnifiedPhonemePipeline,
    path_bids=path_bids,
    path_output=path_output,
    path_results=path_results,
    methods = ['high_gamma', 'adaptive_bands'], # 'multi_band', 'spectral', 'wavelet', 'mfcc'], #['high_gamma'], #, 'multi_band', 'spectral', 'wavelet', 'mfcc'], #, 'mfcc'],  'spectral',  # Start with these 3
    pca_components=30,
    debug_mode=False
)


Testing feature extraction method: high_gamma

PhoneticDictionary: Initialized with DEBUG_MODE=False
UnifiedPhonemePipeline: Pipeline initialized: high_gamma, PCA=30, groups=True
UnifiedPhonemePipeline: No checkpoint found for high_gamma with PCA=30
Creating new pipeline for high_gamma
RUNNING COMPLETE PIPELINE FOR high_gamma
UnifiedPhonemePipeline: Running steps 1-6...
UnifiedPhonemePipeline: No checkpoint found for high_gamma with PCA=30
UnifiedPhonemePipeline: Step 1: Initializing
CustomBrainAudioDecoder: Initializing CustomBrainAudioDecoder with debug_mode=False
Step 1: Decoder initialized with PCA components=30
UnifiedPhonemePipeline: Step 2: Setting up acoustic change detector
CustomBrainAudioDecoder: 
Participant stratification results:
CustomBrainAudioDecoder:   Participants with most relevant channels: 3
CustomBrainAudioDecoder:   Participants with relevant channels: 4
CustomBrainAudioDecoder:   Participants with least relevant channels: 3
CustomBrainAudioDecoder: 
Top partic

AcousticChangeDetector: Accumulated 32 phoneme segments so far
AcousticChangeDetector: Processing batch 2/3
AcousticChangeDetector: Accumulated 64 phoneme segments so far
AcousticChangeDetector: Processing batch 3/3
AcousticChangeDetector: Accumulated 96 phoneme segments so far
AcousticChangeDetector: Accumulated data from 3 batches:
AcousticChangeDetector: Total phoneme segments: 96
AcousticChangeDetector: Unique phonemes: 29
UnifiedPhonemePipeline: Successfully accumulated 96 test samples
UnifiedPhonemePipeline: Step 6: Validating phonemes
PhonemeValidator: Initialized with DEBUG_MODE=False
PhonemeValidator: Debug mode enabled
UnifiedPhonemePipeline: ✓ Step 6: Validator initialized
UnifiedPhonemePipeline: Resolving 99 unknown phonemes in training...
PhonemeValidator [DEBUG]: Attempting to resolve 99 unknown phonemes
PhonemeValidator: Resolved phoneme at position 0 in 'dakker' as 'd'
PhonemeValidator: Resolved phoneme at position 1 in 'dakker' as 'ɑ'
PhonemeValidator: Resolved phoneme

SimplePhonemeModels: naive_bayes training accuracy: 0.6495
SimplePhonemeModels: Training logistic...
SimplePhonemeModels: logistic training accuracy: 0.8946
SimplePhonemeModels: Training lda...
SimplePhonemeModels: lda training accuracy: 0.8946
SimplePhonemeModels: Training knn...
SimplePhonemeModels: knn training accuracy: 0.7843
SimplePhonemeModels: Training linear_svm...
SimplePhonemeModels: linear_svm training accuracy: 0.8946
SimplePhonemeModels: Training random_forest...
SimplePhonemeModels: random_forest training accuracy: 0.8946
SimplePhonemeModels: Training gradient_boosting...
SimplePhonemeModels: gradient_boosting training accuracy: 0.8946
SimplePhonemeModels: Training lightgbm...
SimplePhonemeModels: lightgbm training accuracy: 0.8946
SimplePhonemeModels: Training mlp...


C:\Users\irina\anaconda3\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


SimplePhonemeModels: mlp training accuracy: 0.6656
SimplePhonemeModels: mlp training accuracy: 0.6656
SimplePhonemeModels: mlp validation accuracy: 0.6707
SimplePhonemeModels: Training cnn...


...

SimplePhonemeModels: mlp training accuracy: 0.4313
SimplePhonemeModels: mlp training accuracy: 0.4313
SimplePhonemeModels: mlp validation accuracy: 0.4625
SimplePhonemeModels: Training cnn...


C:\Users\irina\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


SimplePhonemeModels: cnn training accuracy: 0.7312
SimplePhonemeModels: cnn training accuracy: 0.7312
SimplePhonemeModels: cnn validation accuracy: 0.6125
SimplePhonemeModels: Training rnn...


C:\Users\irina\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


SimplePhonemeModels: rnn training accuracy: 0.8313
SimplePhonemeModels: rnn training accuracy: 0.8313
SimplePhonemeModels: rnn validation accuracy: 0.6500
SimplePhonemeModels: Training tcn...
SimplePhonemeModels: tcn training accuracy: 0.7969
SimplePhonemeModels: tcn training accuracy: 0.7969
SimplePhonemeModels: tcn validation accuracy: 0.6375
Evaluating models
SimplePhonemeModels: naive_bayes test accuracy: 0.1771
SimplePhonemeModels: logistic test accuracy: 0.1354
SimplePhonemeModels: lda test accuracy: 0.1667
SimplePhonemeModels: knn test accuracy: 0.1042
SimplePhonemeModels: linear_svm test accuracy: 0.2083
SimplePhonemeModels: random_forest test accuracy: 0.1979
SimplePhonemeModels: gradient_boosting test accuracy: 0.1875
SimplePhonemeModels: lightgbm test accuracy: 0.1458
SimplePhonemeModels: Warning: Shape mismatch for mlp
SimplePhonemeModels: Expected shape: (None, 296, 30), Got shape: (96, 295, 30)
SimplePhonemeModels: mlp test accuracy: 0.1979
SimplePhonemeModels: Warning: S

##### 